# IOAI 2026 — Home Task 1 (Operation Night Watch)

Solution code generated by **Gemma 4 (gemma-4-31b-it)** under contest-sim constraints: fresh chat per prompt, `maxOutputTokens=2000`, code-only replies. Assembled verbatim from the drill transcript (only glue: setup cell + name aliases). 2026-07-12.

In [ ]:
# Setup (starter boilerplate — mirrors the official Home-Task-1 notebook)
import os
import pandas as pd, numpy as np, torch, librosa
from transformers import ASTFeatureExtractor, ASTForAudioClassification

# auto-discover the data root (dir containing train.csv) under /kaggle/input
DATA = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'train.csv' in files and 'fine_tune.csv' in files:
        DATA = root
        break
print('DATA =', DATA)
print(os.listdir(DATA))
MODEL_DIR = os.path.join(DATA, 'model')
print('model dir exists:', os.path.isdir(MODEL_DIR), os.listdir(MODEL_DIR) if os.path.isdir(MODEL_DIR) else '')
feature_extractor = ASTFeatureExtractor.from_pretrained(MODEL_DIR)
model = ASTForAudioClassification.from_pretrained(MODEL_DIR)
print(model.classifier)

**P1 — extend classifier head 16 → 29 (copy old weights)** — Gemma's first attempt used `.in_features` on `ASTMLPHead` (crash); traceback pasted back, this is Gemma's fix (FIX1).

In [ ]:
import torch.nn as nn

old_dense = model.classifier.dense
in_features = old_dense.in_features
new_dense = nn.Linear(in_features, 29)

with torch.no_grad():
    new_dense.weight[:16].copy_(old_dense.weight)
    new_dense.bias[:16].copy_(old_dense.bias)

model.classifier.dense = new_dense

**P2 — stratified 85/15 train/val split**

In [ ]:
from sklearn.model_selection import train_test_split

df_old = pd.read_csv(f"{DATA}/train.csv")
df_new = pd.read_csv(f"{DATA}/fine_tune.csv")

old_train, old_val = train_test_split(
    df_old, test_size=0.15, stratify=df_old['target'], random_state=0
)
new_train, new_val = train_test_split(
    df_new, test_size=0.15, stratify=df_new['target'], random_state=0
)

**P3 — Dataset + collate** — first attempt double-prefixed `audio/` (CSV paths already include it); this is Gemma's FIX3.

In [ ]:
from torch.utils.data import Dataset

class AudioDataset(Dataset):
    def __init__(self, df):
        self.df = df

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        audio_path = f"{DATA}/{row['path']}"
        waveform, _ = librosa.load(audio_path, sr=16000)
        return waveform, int(row['target'])

def collate_fn(batch):
    waveforms, labels = zip(*batch)
    inputs = feature_extractor(
        list(waveforms),
        sampling_rate=16000,
        return_tensors="pt",
        padding=True
    )
    return inputs.input_values, torch.tensor(labels, dtype=torch.long)

In [ ]:
# Aliases so later Gemma cells (which assume AudioDS/collate) match P3's names
AudioDS = AudioDataset
collate = collate_fn

**P4 — fine-tune with experience replay (dual LR)** — optimizer line originally used `model.ast` (crash); replaced verbatim with Gemma's FIX2.

In [ ]:
model.to('cuda')
optimizer = torch.optim.AdamW([
    {'params': model.audio_spectrogram_transformer.parameters(), 'lr': 1e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-3}
])
criterion = torch.nn.CrossEntropyLoss()
model.train()

for epoch in range(3):
    resampled_old = old_train.sample(n=len(new_train), replace=True)
    train_df = pd.concat([new_train, resampled_old])
    loader = torch.utils.data.DataLoader(AudioDS(train_df), batch_size=8, collate_fn=collate, shuffle=True)
    epoch_loss = 0
    for input_values, labels in loader:
        input_values, labels = input_values.to('cuda'), labels.to('cuda')
        optimizer.zero_grad()
        logits = model(input_values=input_values).logits
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    print(f"Epoch {epoch+1} Mean Loss: {epoch_loss/len(loader):.4f}")

**P5 — validation accuracies**

In [ ]:
from torch.utils.data import DataLoader

def evaluate(df):
    dl = DataLoader(AudioDS(df), batch_size=8, collate_fn=collate, shuffle=False)
    all_preds, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for input_values, labels in dl:
            logits = model(input_values=input_values.to('cuda')).logits
            all_preds.append(logits.argmax(dim=-1).cpu().numpy())
            all_labels.append(labels.numpy())
    return np.concatenate(all_preds), np.concatenate(all_labels)

p_old, t_old = evaluate(old_val)
p_new, t_new = evaluate(new_val)

acc_old = (p_old == t_old).mean()
acc_new = (p_new == t_new).mean()

print(f"Old Classes Accuracy: {acc_old:.4f}")
print(f"New Classes Accuracy: {acc_new:.4f}")
print(f"Mean Accuracy: {(acc_old + acc_new) / 2:.4f}")

**P6 — predict hidden test + write submission.csv**

In [ ]:
df_sub = pd.read_csv(f"{DATA}/submission.csv")
dl = torch.utils.data.DataLoader(AudioDS(df_sub), batch_size=8, shuffle=False, collate_fn=collate)
model.eval()
device = next(model.parameters()).device
preds = []

with torch.no_grad():
    for batch in dl:
        inputs, _ = batch
        inputs = torch.as_tensor(inputs).to(device)
        logits = model(input_values=inputs).logits
        preds.extend(logits.argmax(dim=-1).cpu().tolist())

df_sub['target'] = preds
df_sub.to_csv('submission.csv', index=False)